# SP2 – příklad použití a testování (2026)

Pravidla vyhodnocení (stejná jako v referenčních datech `*.ref-1.pkl`, `*.ref-2.pkl`):
* vzdálenost = Levenshteinova vzdálenost (záměna, vložení i vynechání znaku = 1), "ch" = dva znaky
* slovo = posloupnost znaků oddělená mezerami
  (např. `e-mailem`, `prokáže-li`, `16,7` jsou jedno slovo)
* `find_similar` vrací množinu slov ve vzdálenosti přesně `distance`
* hledané slovo se ve vstupním textu nemusí vyskytovat


In [10]:
# import functions initialize and find_similar
from users.SP2_interface import initialize, find_similar

# or make your own
'''
def initialize(filepath, coding = "cp1250"):
    all_words = {}
    all_words['example'] = 999
    return all_words

def find_similar(word, data, distance = 0):
    similar_words = set()
    return similar_words
'''

'\ndef initialize(filepath, coding = "cp1250"):\n    all_words = {}\n    all_words[\'example\'] = 999\n    return all_words\n\ndef find_similar(word, data, distance = 0):\n    similar_words = set()\n    return similar_words\n'

In [11]:
# read all data to object data
import time

start = time.time()
data = initialize("CZ_KKY_APK_dump.txt", "utf-8")
print("počet slov:", len(data), " - doba načtení:", time.time() - start)
#print("all words", data.keys())

#test
similar = find_similar("tabulka", data, distance = 1)
print("podobná slova:", similar)

UnboundLocalError: cannot access local variable 'line' where it is not associated with a value

In [4]:
# load test/reference data
import pickle

ref_files = ["CZ_KKY_APK_dump.ref-1.pkl", "CZ_KKY_APK_dump.ref-2.pkl"]
refs = {}
for ref_file in ref_files:
    with open(ref_file, "rb") as f:
        refs[ref_file] = pickle.load(f)
    print(ref_file, "- počet dotazů:", len(refs[ref_file]))

# ukázka: klíč je (hledané slovo, vzdálenost), hodnota je množina nalezených slov
ref = refs[ref_files[0]]
print(list(ref.items())[:5])
print(ref[("sazky", 2)])

CZ_KKY_APK_dump.ref-1.pkl - počet dotazů: 107
CZ_KKY_APK_dump.ref-2.pkl - počet dotazů: 107
[(('jejich', 2), {'jich'}), (('viděl', 2), {'vydal'}), (('úhlopříčkou', 5), {'hlavičkou'}), (('první', 2), {'právník', 'paní', 'trvání', 'přání'}), (('připravit', 4), {'přidali', 'připadá', 'proplatit'})]
{'taky'}


## final test

In [5]:
# test my code
for ref_file, ref in refs.items():
    chyb = 0
    cas = 0.0
    for word, dist in ref:
        start = time.time()
        similar = find_similar(word, data, distance = dist)
        cas += time.time() - start
        #print(word, dist, similar)
        chyb_local = ref[(word, dist)].symmetric_difference(similar)   # prvky jen v jedné z množin
        if len(chyb_local) > 0:
            print("\nchyba    :", word, dist)
            print("nalezeno :", similar)
            print("správně  :", ref[(word, dist)])
        chyb += len(chyb_local)
    print("\n%s - počet chyb: %d, čas find_similar: %.4f s\n" % (ref_file, chyb, cas))


chyba    : jejich 2
nalezeno : set()
správně  : {'jich'}

chyba    : viděl 2
nalezeno : set()
správně  : {'vydal'}

chyba    : úhlopříčkou 5
nalezeno : set()
správně  : {'hlavičkou'}

chyba    : první 2
nalezeno : set()
správně  : {'právník', 'paní', 'trvání', 'přání'}

chyba    : připravit 4
nalezeno : set()
správně  : {'přidali', 'připadá', 'proplatit'}

chyba    : jednatřicetiletého 7
nalezeno : set()
správně  : {'jednatřicet', 'jedenaosmdesátiletého'}

chyba    : nemocenské 5
nalezeno : set()
správně  : {'zemědělské', 'mateřské', 'nemoci', 'neplacené'}

chyba    : jedenatřicítky 7
nalezeno : set()
správně  : {'jedničky'}

chyba    : přehrávačích 6
nalezeno : set()
správně  : {'nahrávač'}

chyba    : apple 0
nalezeno : set()
správně  : {'apple'}

chyba    : zaměstnancům 6
nalezeno : set()
správně  : {'poslancům'}

chyba    : es 1
nalezeno : set()
správně  : {'s'}

chyba    : nechám 2
nalezeno : set()
správně  : {'nechat', 'nechává', 'nechci', 'nemám', 'nechal'}

chyba    : práci 0
